In [1]:
import numpy as np
import pandas as pd


J1239-2041, J1946-1318, J2049+3351,  is not isolated so ignore for now (butt load of stars next to them)
J1506-1125 has a red star near it so idk how isolated it is
J0056-0611 also has some star or something near by
J0923+3028 has like 3 things near it
J1832+2031 has some stuff near by

In [2]:
#proper motion in mas/yrs
               # J2322+0509 (1), J0935+4411(2), J1239-2041(3), J0634+3803(4), J1946-1318(5), J0338-8139(6), J1526-2711(7), J2049+3351(8), J1506-1125(9), J1235+1543(10), J0056-0611(11),   J0923+3028(12), J1436+5010(13), J1832+2031(14)
proper_motiona = [11.758,       -20.315,        -9.039,        -23.590,       4.619,         13.029,        2.668,         -0.446,        0.419,         -25.486,        16.746,         -3.797,           8.558,          8.116]
proper_motiong = [-6.965,       -1.213,         7.746,         -29.729,       0.744,         5.596,         -2.237,        -2.176,        -10.712,       -32.511,        -15.374,        -29.642,          -5.140,         -0.329]
#all the propermotions for RA and Dec from Sinbad

In [3]:
binary_data = pd.read_csv('/Users/leighharwell2/Desktop/DW/IRchara/list_spherex_binaries.txt', sep=r'\s+')
print(binary_data)
#load in the data as a table using pandas

          Name     Gmag  Gmag_e            ra            dec
0   J2322+0509  18.7466  0.0020  23:22:30.203    5:09:42.058
1   J0935+4411  17.8045  0.0062   9:35:06.897   44:11:06.919
2   J1239-2041  18.9418  0.0019  12:39:50.370  -20:41:42.276
3   J0634+3803  17.1331  0.0011   6:34:49.904   38:03:51.975
4   J1946-1318  17.4439  0.0015  19:46:12.798  -13:18:22.786
5   J0338-8139  17.2902  0.0011   3:38:16.193  -81:39:29.967
6   J1526-2711  18.3489  0.0020  15:26:01.115  -27:11:56.660
7   J2049+3351  18.7010  0.0024  20:49:51.274   33:51:53.126
8   J1506-1125  17.0104  0.0015  15:06:12.345  -11:25:11.994
9   J1235+1543  17.5162  0.0012  12:35:49.853   15:43:18.954
10  J0056-0611  17.5441  0.0010   0:56:48.240   -6:11:41.804
11  J0923+3028  15.9203  0.0008   9:23:45.598   30:28:04.609
12  J1436+5010  18.4158  0.0014  14:36:33.303   50:10:26.843
13  J1832+2031  17.6301  0.0007  18:32:36.539   20:31:08.202


In [4]:
#add the proper motion data to the dataframe
binary_data['pm_ra'] = proper_motiona
binary_data['pm_dec'] = proper_motiong
print(binary_data.head())

         Name     Gmag  Gmag_e            ra            dec   pm_ra  pm_dec
0  J2322+0509  18.7466  0.0020  23:22:30.203    5:09:42.058  11.758  -6.965
1  J0935+4411  17.8045  0.0062   9:35:06.897   44:11:06.919 -20.315  -1.213
2  J1239-2041  18.9418  0.0019  12:39:50.370  -20:41:42.276  -9.039   7.746
3  J0634+3803  17.1331  0.0011   6:34:49.904   38:03:51.975 -23.590 -29.729
4  J1946-1318  17.4439  0.0015  19:46:12.798  -13:18:22.786   4.619   0.744


In [5]:
from astropy.coordinates import SkyCoord
import astropy.units as u

coords = SkyCoord(ra=binary_data['ra'], dec=binary_data['dec'], unit=(u.hourangle, u.deg)) 
#turning the ra and dec into degrees for convertion
binary_data['ra'] = coords.ra.deg

binary_data['dec'] = coords.dec.deg

In [6]:
#now finding the new Declination and Right Ascension for each binary star system after 10 years of proper motion
#t = 10.66 #time in years

def update_table_positions(dataframe, years):
  """Accepts a table (DataFrame) and calculates new positions for all stars at once.

  Parameters:
  dataframe (DataFrame): Your existing table
  years (float): Number of years to project into the future
  """
  # cope the dataframe to avoid modifying the original
  df_result = dataframe.copy()

  # (fixes str -> float issues)
  numeric_cols = ['ra', 'dec', 'pm_ra', 'pm_dec']
  for col in numeric_cols:
      df_result[col] = pd.to_numeric(df_result[col], errors='coerce')

  # Conversion factor: 1 degree = 3,600,000 milliarcseconds
  MAS_TO_DEG = 1.0 / 3_600_000

  # new RA = old RA + (pm_ra * years * dagree convertion) / cos(dec)
  df_result['new_ra'] = df_result['ra'] + (
      df_result['pm_ra'] * years * MAS_TO_DEG
  ) / np.cos(np.radians(df_result['dec']))
  # new Dec = old Dec + (pm_dec * years * degree conversion)
  df_result['new_dec'] = df_result['dec'] + (
      df_result['pm_dec'] * years * MAS_TO_DEG
  )

  # Keep all RA values cleanly wrapped between 0 and 360 degrees
  df_result['new_ra'] = df_result['new_ra'] % 360.0

  return df_result





$ RA_c = RA + \mu(t * 1/3,600,000) $

$ Dec_c = Dec + \mu (t * 1/3,600,000) $

In [7]:

#apply function to the data set
updated_table = update_table_positions(binary_data, years=13.0)

print(updated_table)


          Name     Gmag  Gmag_e          ra        dec   pm_ra  pm_dec  \
0   J2322+0509  18.7466  0.0020  350.625846   5.161683  11.758  -6.965   
1   J0935+4411  17.8045  0.0062  143.778738  44.185255 -20.315  -1.213   
2   J1239-2041  18.9418  0.0019  189.959875 -20.695077  -9.039   7.746   
3   J0634+3803  17.1331  0.0011   98.707933  38.064437 -23.590 -29.729   
4   J1946-1318  17.4439  0.0015  296.553325 -13.306329   4.619   0.744   
5   J0338-8139  17.2902  0.0011   54.567471 -81.658324  13.029   5.596   
6   J1526-2711  18.3489  0.0020  231.504646 -27.199072   2.668  -2.237   
7   J2049+3351  18.7010  0.0024  312.463642  33.864757  -0.446  -2.176   
8   J1506-1125  17.0104  0.0015  226.551437 -11.419998   0.419 -10.712   
9   J1235+1543  17.5162  0.0012  188.957721  15.721932 -25.486 -32.511   
10  J0056-0611  17.5441  0.0010   14.201000  -6.194946  16.746 -15.374   
11  J0923+3028  15.9203  0.0008  140.939992  30.467947  -3.797 -29.642   
12  J1436+5010  18.4158  0.0014  219.1

In [8]:
#apply function to the data set
updated_table = update_table_positions(binary_data, years=22.0)

print(updated_table)

          Name     Gmag  Gmag_e          ra        dec   pm_ra  pm_dec  \
0   J2322+0509  18.7466  0.0020  350.625846   5.161683  11.758  -6.965   
1   J0935+4411  17.8045  0.0062  143.778738  44.185255 -20.315  -1.213   
2   J1239-2041  18.9418  0.0019  189.959875 -20.695077  -9.039   7.746   
3   J0634+3803  17.1331  0.0011   98.707933  38.064437 -23.590 -29.729   
4   J1946-1318  17.4439  0.0015  296.553325 -13.306329   4.619   0.744   
5   J0338-8139  17.2902  0.0011   54.567471 -81.658324  13.029   5.596   
6   J1526-2711  18.3489  0.0020  231.504646 -27.199072   2.668  -2.237   
7   J2049+3351  18.7010  0.0024  312.463642  33.864757  -0.446  -2.176   
8   J1506-1125  17.0104  0.0015  226.551437 -11.419998   0.419 -10.712   
9   J1235+1543  17.5162  0.0012  188.957721  15.721932 -25.486 -32.511   
10  J0056-0611  17.5441  0.0010   14.201000  -6.194946  16.746 -15.374   
11  J0923+3028  15.9203  0.0008  140.939992  30.467947  -3.797 -29.642   
12  J1436+5010  18.4158  0.0014  219.1

In [9]:
#apply function to the data set
updated_table = update_table_positions(binary_data, years=26.0)

print(updated_table)

          Name     Gmag  Gmag_e          ra        dec   pm_ra  pm_dec  \
0   J2322+0509  18.7466  0.0020  350.625846   5.161683  11.758  -6.965   
1   J0935+4411  17.8045  0.0062  143.778738  44.185255 -20.315  -1.213   
2   J1239-2041  18.9418  0.0019  189.959875 -20.695077  -9.039   7.746   
3   J0634+3803  17.1331  0.0011   98.707933  38.064437 -23.590 -29.729   
4   J1946-1318  17.4439  0.0015  296.553325 -13.306329   4.619   0.744   
5   J0338-8139  17.2902  0.0011   54.567471 -81.658324  13.029   5.596   
6   J1526-2711  18.3489  0.0020  231.504646 -27.199072   2.668  -2.237   
7   J2049+3351  18.7010  0.0024  312.463642  33.864757  -0.446  -2.176   
8   J1506-1125  17.0104  0.0015  226.551437 -11.419998   0.419 -10.712   
9   J1235+1543  17.5162  0.0012  188.957721  15.721932 -25.486 -32.511   
10  J0056-0611  17.5441  0.0010   14.201000  -6.194946  16.746 -15.374   
11  J0923+3028  15.9203  0.0008  140.939992  30.467947  -3.797 -29.642   
12  J1436+5010  18.4158  0.0014  219.1